In [1]:

import os
os.chdir('/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_11_scripts/')

import scipy.io
import h5py
import numpy as np
import sys
import hc11_BaseFunctions as hc11_bf
import normalizing_functions as norm

from scipy import signal as sig
from scipy import stats as stats
import math
import pandas as pd
import csv
import matplotlib.pyplot as plt
from scipy import signal
import seaborn as sns

%matplotlib widget

sessions = ['Achilles_10252013','Buddy_06272013','Cicero_09012014','Gatsby_08022013','Achilles_11012013','Cicero_09102014','Cicero_09172014','Gatsby_08282013']

rejected_shanks = np.load('rejected_shanks.npy',allow_pickle=True).item()
rejected_electrodes = np.load('rejected_electrodes.npy',allow_pickle=True).item()


In [2]:
for session in sessions:
    print(session)
    all_shanks,left_shanks,right_shanks = hc11_bf.get_shanks_info(session)
    LFP,srate,SWS,SWS_lfp_index,SWS_lfp_index_pre,SWS_lfp_index_pos = hc11_bf.load_lfp_hc11_data(session)
    
    lfp_index = SWS_lfp_index[0:int(60*60*srate)]
        
    shanks_electrode_position = []
    electrode_position = np.arange(-9,10,1)
    
    for sk in range(len(all_shanks)):
        shank = all_shanks[sk]
        
        max_ripple_amp = []
        for ch_counter,ch in enumerate(shank):
            print(ch)
            ripple = hc11_bf.eegfilt(LFP[ch,lfp_index],srate,100,250)
            ripple_amp = np.abs(hc11_bf.hilbert(ripple))
            ind = hc11_bf.detect_peaks(ripple_amp,mph = 2*np.nanstd(ripple_amp),mpd=0.1*srate)
            max_ripple_amp.append(np.nanmean(ripple_amp[ind]))
        max_ripple_amp = np.array(max_ripple_amp)
        
        begin_position = np.argmax(max_ripple_amp)
        norm_electrode_position = begin_position - electrode_position
        
        elec_pos = []
        for ch in range(len(shank)):
            elec_pos.append(electrode_position[np.where(norm_electrode_position == ch)[0]][0]*20)
        shanks_electrode_position.append(elec_pos)
    shanks_electrode_position = np.array(shanks_electrode_position)
    
    os.chdir('/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_11_scripts/')
    np.save('electrode_position_' + session, shanks_electrode_position)


Achilles_10252013
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
Buddy_06272013
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
Cicero_09012014
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
64
65

In [ ]:
output = np.load(f'electrode_position_{session}.npy')
output

In [106]:
# this is the old way
# get all heights and plot a mean comodul

shanks_electrode_position = []
electrode_position = np.arange(-9,10,1)

for sk in range(len(all_shanks)):
    shank = all_shanks[sk]
    ripple_ref = norm.z_score_norm(LFP[shank[4],lfp_index])
    ripple_ref = hc11_bf.eegfilt(ripple_ref,srate,100,250)
    # ripple = bf11.eegfilt(ripple,srate,150,300)
    ripple_amp = np.abs(hc11_bf.hilbert(ripple_ref))

    ind = hc11_bf.detect_peaks(ripple_amp,mph = 2*np.nanstd(ripple_amp),mpd=0.1*srate)

    window = int(0.5*srate)
    ripple_triggered_channels = np.zeros([len(shank),2*window+1])

    for ch_counter,ch in enumerate(shank):
        print(ch)
        ripple = hc11_bf.eegfilt(LFP[ch,lfp_index],srate,100,250)
        ripple_amp = np.abs(hc11_bf.hilbert(ripple))

        ripple_triggered = np.zeros([2*window+1])
        counter = 0
        for gg in range(0,ind.shape[0]):
            if(ind[gg] > window) & (ind[gg] < ripple.shape[0] - window):
                ripple_window = np.arange(ind[gg] - window, ind[gg] + window + 1).astype(int)
                ripple_triggered += ripple_amp[ripple_window]
                counter += 1
        ripple_triggered_channels[ch_counter,:] = ripple_triggered/counter
    
    max_ripple_amp = np.nanmax(ripple_triggered_channels,1)

    begin_position = np.argmax(max_ripple_amp)
    norm_electrode_position = begin_position - electrode_position
    
    elec_pos = []
    for ch in range(len(shank)):
        elec_pos.append(electrode_position[np.where(norm_electrode_position == ch)[0]][0]*20)
    shanks_electrode_position.append(elec_pos)
shanks_electrode_position = np.array(shanks_electrode_position)


os.chdir('/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_11_scripts/')
np.save('electrode_position_' + session, shanks_electrode_position)



0
1
2
3
4
5
6
7
8
9


KeyboardInterrupt: 

In [ ]:

    ripple_ref = norm.z_score_norm(LFP[shank[4],lfp_index])
    ripple_ref = hc11_bf.eegfilt(ripple_ref,srate,100,250)
    # ripple = bf11.eegfilt(ripple,srate,150,300)
    ripple_amp = np.abs(hc11_bf.hilbert(ripple_ref))

    ind = hc11_bf.detect_peaks(ripple_amp,mph = 2*np.nanstd(ripple_amp),mpd=0.1*srate)

    window = int(0.5*srate)
    ripple_triggered_channels = np.zeros([len(shank),2*window+1])


In [ ]:
lfp_index = SWS_lfp_index[0:int(60*60*srate)]

# get all heights and plot a mean comodul

shanks_electrode_position = []
electrode_position = np.arange(-9,10,1)

for sk in range(len(all_shanks)):
# for sk in range(1):
    
    shank = all_shanks[sk]
    
    max_ripple_amp = []
    for ch_counter,ch in enumerate(shank):
        print(ch)
        ripple = hc11_bf.eegfilt(LFP[ch,lfp_index],srate,100,250)
        ripple_amp = np.abs(hc11_bf.hilbert(ripple))
        ind = hc11_bf.detect_peaks(ripple_amp,mph = 2*np.nanstd(ripple_amp),mpd=0.1*srate)
        max_ripple_amp.append(np.nanmean(ripple_amp[ind]))
    max_ripple_amp = np.array(max_ripple_amp)
    
    begin_position = np.argmax(max_ripple_amp)
    norm_electrode_position = begin_position - electrode_position
    
    elec_pos = []
    for ch in range(len(shank)):
        elec_pos.append(electrode_position[np.where(norm_electrode_position == ch)[0]][0]*20)
    shanks_electrode_position.append(elec_pos)
shanks_electrode_position = np.array(shanks_electrode_position)

os.chdir('/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_11_scripts/')
np.save('electrode_position_' + session, shanks_electrode_position)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


In [115]:
shanks_electrode_position

array([[  40,   20,    0,  -20,  -40,  -60,  -80, -100, -120, -140],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [  20,    0,  -20,  -40,  -60,  -80, -100, -120, -140, -160],
       [ 100,   80,   60,   40,   20,    0,  -20,  -40,  -60,  -80],
       [ 100,   80,   60,   40,   20,    0,  -20,  -40,  -60,  -80],
       [  60,   40,   20,    0,  -20,  -40,  -60,  -80, -100, -120],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [   0,  -20,  -40,  -60,  -80, -100, -120, -140, -160, -180],
       [  40,   20,    0,  -20,  -40,  -60,  -80, -100, -120, -140]])